```
===========================================================================================
# *                ███████╗██████╗ ██╗  ██╗██╗██████╗ ██╗████████╗
# *                ██╔════╝██╔══██╗██║  ██║██║██╔══██╗██║╚══██╔══╝
# *                ███████╗██████╔╝███████║██║██████╔╝██║   ██║
# *                ╚════██║██╔═══╝ ██╔══██║██║██╔══██╗██║   ██║
# *                ███████║██║     ██║  ██║██║██║  ██║██║   ██║
# *                ╚══════╝╚═╝     ╚═╝  ╚═╝╚═╝╚═╝  ╚═╝╚═╝   ╚═╝
# * — Smoothed Particle Hydrodynamics Integrated Resource for Instruction and Teaching ─
===========================================================================================
```

# ────────────────────────────────────────────────────────────────────
# Cellule 1 - Configuration (seule cellule a modifié avant de lancer)
# ────────────────────────────────────────────────────────────────────

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
from io_.widgets_config import _display_widgets

# ─── Implémentation élément d'inteface ───────────────
_display_widgets()
# ───────────────────────────────────────────────────────────

# ────────────────────────────────────────────────────────────────────
# Cellule 2 - Chargement des propriétés
# ────────────────────────────────────────────────────────────────────

In [ ]:
from constant import physical_properties as phys
from constant import numerical_properties as num
from constant import sph_properties as sph
from constant import ale_properties as ale

print(f'rho0={phys.rho0} kg/m³  dx={sph.dx} m  h={sph.h:4f} m')
print(f'mesh_mode={ale.mesh_mode}  mode_ale={ale.mode_ale}')

# ────────────────────────────────────────────────────────────────────
# Cellule 3 - Initialisation du cas
# ────────────────────────────────────────────────────────────────────

In [ ]:
import importlib
import system.control_dict as ctrl
case_module = importlib.import_module(f'cases.{ctrl.test_case}.setup')
state = case_module.init(
    vars(ctrl),vars(phys),vars(sph),vars(ale),vars(num))

print(f'Initialisation OK : {len(state["pos"])} particules')
print(f'  dont fluides : {int((state["types"]==0).sum())}')
print(f'  dt initial   : {state["dt"]:.2e} s')
print(f'  Nombre de pas estimé : {int(ctrl.finaltime/state["dt"]):,}')

# ────────────────────────────────────────────────────────────────────
# Cellule 4 - Lancement de la simulation
# ────────────────────────────────────────────────────────────────────

In [ ]:
from src.Solver import run
import time
t_start = time.perf_counter()
run(state,vars(ctrl),vars(phys),vars(sph))
t_end = time.perf_counter()
print(f"Durée de simulation : {t_end - t_start:.1f} s")



# ────────────────────────────────────────────────────────────────────
# Cellule 5 - visualisation rapide
# ────────────────────────────────────────────────────────────────────

In [ ]:
%matplotlib inline
import matplotlib
matplotlib.use('inline')
import matplotlib.pyplot as plt
import numpy as np

# Afficher la position des particules à t=0
pos_final = state['pos']
types_final = state['types']
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(pos_final[types_final==0, 0], pos_final[types_final==0, 1],
           s=2, c='steelblue', label='Fluide', alpha=0.6)
ax.scatter(pos_final[types_final==1, 0], pos_final[types_final==1, 1],
           s=2, c='gray', label='Mur', alpha=0.4)
ax.set_aspect('equal')
ax.legend(); ax.set_title(f'État final — {ctrl.test_case}')
plt.tight_layout(); plt.show()